# Colab setup: Google Drive workspace + data

Run top to bottom at the start of every Colab session.

**First session** (~1 h): download RDD2022 + MWPD, extract, run the Phase-1 pipeline (`make data`),
then save `data/processed/` to Drive as `cache/processed.tar` (only the images used by the split
lists, a few GB).
**Later sessions** (a few minutes): the cache is restored to local disk and the download,
extraction and pipeline steps are skipped.

The cache is tagged with the last commit that changed `src/data/`, `configs/data.yaml`,
`configs/rain.yaml` or the split lists. If any of these changed, the cache is treated as stale and
the full pipeline runs again (this needs the RDD2022 zip, so do not delete it while the data
pipeline is still changing).

| Where | What | Why |
|---|---|---|
| `MyDrive/<DRIVE_DIR>/cache/` | `processed.tar` + `processed.stamp` | restores `data/processed/` in minutes |
| `MyDrive/<DRIVE_DIR>/downloads/` | RDD2022 zip (13.3 GB) | only needed to rebuild the cache |
| `MyDrive/<DRIVE_DIR>/raw/mwpd/` | MWPD images | small, persists |
| `MyDrive/<DRIVE_DIR>/outputs/` | logs, training runs | persists |
| `/content/deep-learning-cse365/` | repo + `data/processed` (+ extracted RDD2022 on a rebuild) | local disk is much faster than Drive for many small files |

**Drive quota:** the free plan has 15 GB. The RDD2022 zip alone is 13.3 GB.
If you do not have enough space, set `ZIP_ON_DRIVE = False`: the zip is then downloaded to local disk
whenever the cache has to be rebuilt, and only the cache, MWPD and outputs are kept on Drive.

In [ ]:
# ---- Settings ----
DRIVE_DIR = "cse365_pothole"   # personal folder under MyDrive
ZIP_ON_DRIVE = True            # keep the 13.3 GB RDD2022 zip on Drive (needs ~14 GB free)
USE_CACHE = True               # restore data/processed from Drive if the cache is up to date
REPO_URL = "https://github.com/queso-gato1355/deep-learning-cse365.git"
BRANCH = "main"

In [ ]:
# 1. Mount Drive and create the personal folder
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_DIR
for sub in ["downloads/rdd2022", "raw/mwpd", "outputs/logs", "outputs/runs"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
print("Drive workspace:", DRIVE_ROOT)

In [ ]:
# 2. Clone (or update) the repo on local disk
import subprocess

def sh(*cmd: str) -> None:
    """Run a command, streaming its output into the notebook; raise if it fails."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"`{' '.join(cmd)}` failed with exit code {p.returncode}; see the output above.")

REPO = Path("/content/deep-learning-cse365")
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
%cd {REPO}
!git log --oneline -3

In [ ]:
# 3. Dependencies for the data pipeline.
# Colab already ships numpy/pandas/opencv/matplotlib/seaborn/requests/pyyaml/tqdm.
# requirements.txt pins Python 3.11 + CUDA 12.8 wheels, so it is NOT installed wholesale here.
!pip install -q albumentations==2.0.8 ruff==0.17.3

In [ ]:
# 4. Link persistent Drive folders into the repo layout expected by configs/data.yaml
import os
import shutil

def link(target: Path, link_path: Path) -> None:
    """Replace ``link_path`` with a symlink to ``target``."""
    link_path.parent.mkdir(parents=True, exist_ok=True)
    if link_path.is_symlink():
        link_path.unlink()
    elif link_path.exists():
        shutil.rmtree(link_path)
    os.symlink(target, link_path)

LOCAL_DOWNLOADS = Path("/content/downloads")
ZIP_DIR = (DRIVE_ROOT / "downloads") if ZIP_ON_DRIVE else LOCAL_DOWNLOADS
(ZIP_DIR / "rdd2022").mkdir(parents=True, exist_ok=True)

link(ZIP_DIR, REPO / "data/raw/_downloads")
link(DRIVE_ROOT / "raw/mwpd", REPO / "data/raw/mwpd")
link(DRIVE_ROOT / "outputs/logs", REPO / "outputs/logs")
link(DRIVE_ROOT / "outputs/runs", REPO / "outputs/runs")
# data/raw/rdd2022 and data/processed stay on local disk (fast I/O).
!ls -la data/raw outputs

In [ ]:
# 5. Restore data/processed from the Drive cache if it matches the current data pipeline
import tarfile

CACHE_DIR = DRIVE_ROOT / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TAR = CACHE_DIR / "processed.tar"
CACHE_STAMP = CACHE_DIR / "processed.stamp"
PIPELINE_FILES = ["src/data", "configs/data.yaml", "configs/rain.yaml", "data/processed/splits"]
PIPELINE_REV = subprocess.run(
    ["git", "log", "-1", "--format=%H", "--", *PIPELINE_FILES],
    capture_output=True, text=True, check=True,
).stdout.strip()

CACHED = False
if not USE_CACHE:
    print("USE_CACHE = False: running the full pipeline.")
elif not (CACHE_TAR.exists() and CACHE_STAMP.exists()):
    print("No cache yet: running the full pipeline (first session).")
elif CACHE_STAMP.read_text().strip() != PIPELINE_REV:
    print(f"Cache is stale (built at {CACHE_STAMP.read_text().strip()[:7]}, "
          f"pipeline now at {PIPELINE_REV[:7]}): running the full pipeline.")
else:
    print(f"Restoring {CACHE_TAR} ({CACHE_TAR.stat().st_size / 1e9:.1f} GB) ...")
    with tarfile.open(CACHE_TAR) as tar:
        tar.extractall(REPO, filter="data")
    CACHED = True
    print("Restored data/processed; steps 6-8 will be skipped.")

In [ ]:
# 6. Download the RDD2022 zip (resumable, md5-verified). Skipped if already complete or cached.
import hashlib
import requests
from tqdm.auto import tqdm

RDD_URL = "https://ndownloader.figshare.com/files/38030910"
RDD_ZIP = ZIP_DIR / "rdd2022" / "RDD2022_released_through_CRDDC2022.zip"
RDD_SIZE = 13_264_172_619
RDD_MD5 = "b62bd51d2ffcfaa76c60f234f0cc2bb3"

def md5sum(path: Path) -> str:
    h = hashlib.md5()
    with path.open("rb") as f, tqdm(total=path.stat().st_size, unit="B", unit_scale=True, desc="md5") as bar:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            h.update(chunk)
            bar.update(len(chunk))
    return h.hexdigest()

part = RDD_ZIP.with_suffix(".zip.part")
if CACHED:
    print("Cache restored: download skipped.")
elif RDD_ZIP.exists() and RDD_ZIP.stat().st_size == RDD_SIZE:
    print("Zip already present:", RDD_ZIP)
else:
    free = shutil.disk_usage(ZIP_DIR).free
    have = part.stat().st_size if part.exists() else 0
    if free < RDD_SIZE - have:
        raise RuntimeError(f"Only {free/1e9:.1f} GB free at {ZIP_DIR}; need {(RDD_SIZE-have)/1e9:.1f} GB. "
                           "Free up Drive space or set ZIP_ON_DRIVE = False.")
    headers = {"Range": f"bytes={have}-"} if have else {}
    with requests.get(RDD_URL, headers=headers, stream=True, timeout=60) as r:
        r.raise_for_status()
        mode = "ab" if have and r.status_code == 206 else "wb"
        start = have if mode == "ab" else 0
        with part.open(mode) as f, tqdm(total=RDD_SIZE, initial=start, unit="B", unit_scale=True, desc="RDD2022") as bar:
            for chunk in r.iter_content(chunk_size=1 << 24):
                f.write(chunk)
                bar.update(len(chunk))
    if md5sum(part) != RDD_MD5:
        part.unlink()
        raise ValueError("md5 mismatch: partial file deleted, re-run this cell.")
    part.replace(RDD_ZIP)
    print("Downloaded:", RDD_ZIP)

In [ ]:
# 7. Download MWPD (to Drive, SHA-256 verified, skips files already there)
#    and extract RDD2022 to local disk (/content/.../data/raw/rdd2022, ~13 GB, takes a while).
#    Skipped when the cache was restored. download.py skips a completed extraction
#    (data/raw/rdd2022/.extracted) and resumes an interrupted one.
if CACHED:
    print("Cache restored: download/extraction skipped.")
else:
    sh("python", "-m", "src.data.download", "--dataset", "all")
!du -sh data/raw/rdd2022 data/raw/mwpd/ data/processed 2>/dev/null

## Phase-1 pipeline + cache

Rebuilds `data/processed/` (local disk), the EDA figures/tables and the road-crop dataset
(`make crop`, used by E3/E4), then saves the cache to Drive.
`split.py` regenerates the split lists deterministically (seed 42), overwriting the committed ones;
if they differ from the committed lists the cache is **not** saved (investigate first).

In [ ]:
# 8. Run the pipeline and save data/processed to Drive (skipped when the cache was restored)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
PROC = REPO / "data/processed"

if CACHED:
    print("Cache restored: pipeline skipped.")
else:
    sh("make", "data", "PY=python")
    sh("make", "crop", "PY=python")   # road-region dataset (rdd_crop60) for E3/E4
    changed = subprocess.run(
        ["git", "status", "--porcelain", "data/processed/splits"],
        capture_output=True, text=True, check=True,
    ).stdout
    if changed:
        raise RuntimeError(f"Split lists differ from the committed ones; cache not saved:\n{changed}")
    !git status --short outputs/tables

    # Keep every non-image file, but only the images referenced by a split list.
    used = set()
    for lst in (PROC / "splits").glob("*.txt"):
        used.update((lst.parent / line).resolve() for line in lst.read_text(encoding="utf-8").split())
    files = [p for p in sorted(PROC.rglob("*"))
             if p.is_file() and (p.suffix.lower() not in IMG_EXT or p.resolve() in used)]
    n_img = sum(p.suffix.lower() in IMG_EXT for p in files)
    if n_img != len(used):
        raise RuntimeError(f"{len(used)} images in split lists but {n_img} found on disk.")

    tmp = Path("/content/processed.tar")
    with tarfile.open(tmp, "w") as tar:
        for p in tqdm(files, desc="tar"):
            tar.add(p, arcname=p.relative_to(REPO).as_posix(), recursive=False)
    print(f"{len(files)} files ({n_img} images), {tmp.stat().st_size / 1e9:.2f} GB -> {CACHE_TAR}")
    shutil.copyfile(tmp, CACHE_TAR)
    CACHE_STAMP.write_text(PIPELINE_REV + "\n")
    tmp.unlink()
    print("Cache saved. Next sessions restore it in step 5.")

## Disk check

In [ ]:
!df -h /content /content/drive | cat